In [15]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [16]:
from pathlib import Path

server_dir = Path(os.getcwd()).parent.parent.parent.parent.resolve()
BASE_DATA_DIR = server_dir / "resources/data"

TRAIN_CSV_FILE = "main.csv"
TRAIN_CSV_PATH = os.path.join(BASE_DATA_DIR, TRAIN_CSV_FILE)
TRAIN_CSV_PATH

'E:\\Applications\\projects\\Sehetna Organization\\sehetna\\server\\resources\\data\\main.csv'

## Pipeline Construction for Reproducibility

Assemble all the defined preprocessing steps into a Pipeline. This will ensure that all transformations are applied consistently and reproducibly.


In [17]:
# Custom transformer for date feature engineering
class DateFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()

        countries = X_copy["country_name"].unique()
        country_to_idx = {value: index for index, value in enumerate(countries)}
        X_copy['country_id'] = X_copy['country_name'].map(country_to_idx)

        X_copy['date'] = pd.to_datetime(X_copy['date'])

        X_copy['day_of_week'] = X_copy['date'].dt.dayofweek
        X_copy['quarter'] = X_copy['date'].dt.quarter

        # Add Cyclic Encoding Transformations for dates and seasons.
        X_copy['month_sin'] = np.sin(2 * np.pi * X_copy['month'] / 12)
        X_copy['month_cos'] = np.cos(2 * np.pi * X_copy['month'] / 12)
        X_copy['week_sin'] = np.sin(2 * np.pi * X_copy['week'] / 52)
        X_copy['week_cos'] = np.cos(2 * np.pi * X_copy['week'] / 52)

        # A. Temporal Lags (The "Incubation Period")
        # We look back 1 week for pollution and temprature because diseases aren't instant.
        X_copy['pm25_ugm3_lag_1w'] = X_copy.groupby('country_name')['pm25_ugm3'].shift(1)
        X_copy['pm25_ugm3_lag_2w'] = X_copy.groupby('country_name')['pm25_ugm3'].shift(2)
        X_copy['pm25_ugm3_lag_4w'] = X_copy.groupby('country_name')['pm25_ugm3'].shift(4)

        X_copy['temp_lag_1w'] = X_copy.groupby('country_name')['temperature_celsius'].shift(1)
        X_copy['temp_lag_2w'] = X_copy.groupby('country_name')['temperature_celsius'].shift(2)

        # B. Rolling Averages (The "Chronic Exposure")
        # B1. We calculate the average temperature of the last 4 weeks.
        X_copy['temp_2w_avg'] = (
            X_copy.groupby('country_name')['temperature_celsius'].transform(lambda x: x.rolling(window=2, min_periods=1).mean())
        )

        # B2. Rolling standard deviation (volatility)
        X_copy['temp_2w_volatility'] = (
            X_copy.groupby('country_name')['temperature_celsius'].transform(lambda x: x.rolling(window=2, min_periods=1).std())
        )

        X_copy['temp_4w_volatility'] = (
            X_copy.groupby('country_name')['temperature_celsius'].transform(lambda x: x.rolling(window=4, min_periods=1).std())
        )

        # C. Mathematical Transformations (Non-Linearity)
        # Temperature usually has a U-shaped effect on health (bad if too hot OR too cold)
        X_copy['log_temp_squared'] = np.log1p(X_copy['temperature_celsius'] ** 2)

        # Log-Transform Target Variables
        # We do NOT scale this later.
        X_copy['log_food_security_index'] = np.log1p(X_copy['food_security_index'])


        # D. Scientific Logic: The human body adapts to gradual change.
        # Sudden spikes (Thermal Shock) cause heart attacks and asthma attacks.
        # D1. We calculate: (Current Week Value) - (Last Week Value)
        X_copy['temp_change_rate'] = X_copy.groupby('country_code')['temperature_celsius'].diff()
        X_copy['pm25_change_rate'] = X_copy.groupby('country_code')['pm25_ugm3'].diff()
        X_copy['precip_change_rate'] = X_copy.groupby('country_code')['precipitation_mm'].diff()

        # E. Calculate Spatial Lag for Respiratory Disease
        # We take the average disease rate of the 5 neighbors (excluding the country itself)
        coords = X_copy[['latitude' , 'longitude']].values

        nbrs = NearestNeighbors(n_neighbors=5 , algorithm='ball_tree').fit(coords)
        distances , indices = nbrs.kneighbors(coords)

        neighbor_idx = indices[: , 1:] # Drop the first one (itself)

        X_copy['spatial_lag_pm25'] = X_copy['pm25_ugm3'].values[neighbor_idx].mean(axis=1)
        X_copy['spatial_lag_temp'] = X_copy['temperature_celsius'].values[neighbor_idx].mean(axis=1)
        X_copy['spatial_lag_temp_anomaly'] = X_copy['temp_anomaly_celsius'].values[neighbor_idx].mean(axis=1)

        # F. The "Photochemical Smog" Effect
        # Scientific Logic: High heat catalyzes chemical reactions in pollutants (like Ozone/PM2.5), making them more toxic to the lungs.
        X_copy['pm25_temp_interaction'] = X_copy['pm25_ugm3'] * X_copy['temperature_celsius']
        X_copy['temp_precip_interaction'] = X_copy['temperature_celsius'] * X_copy['precipitation_mm']
        X_copy['pm25_precip_interaction'] = X_copy['pm25_ugm3'] * X_copy['precipitation_mm']

        # G. Hemisphere Indicators
        X_copy['is_northern'] = (X_copy['latitude'] > 0).astype(int)
        X_copy['is_tropical'] = (X_copy['latitude'].abs() < 23.5).astype(int)
        X_copy['distance_to_equator'] = X_copy['latitude'].abs()

        # I. Extreme Weather Impact Score
        X_copy['extreme_weather_score'] = (
            (X_copy['heat_wave_days'] + 1) * 0.4 +
            (X_copy['flood_indicator'] + 1) * 0.3 +
            (X_copy['drought_indicator'] + 1) * 0.3
        )

        # J. The "Vulnerability" Effect
        # Scientific Logic: High pollution is manageable if Healthcare Access is high.
        # It becomes fatal if Healthcare is low.
        # We add a small epsilon (1e-6) to avoid division by zero.

        X_copy['pollution_vulnerability'] = X_copy['pm25_ugm3'] / (X_copy['healthcare_access_index'] + 1e-6)

        X_copy = X_copy.drop(columns=['month', 'week'])

        self.output_columns_ = X_copy.columns.tolist()
        return X_copy

    def get_feature_names_out(self, input_features=None):
        return np.array(self.output_columns_)

# Custom transformer for outlier capping
class IQRCapper(BaseEstimator, TransformerMixin):
    def __init__(self, multiplier=1.5):
        self.multiplier = multiplier
        self.bounds_ = {}  # store per-column bounds

    def fit(self, X, y=None):
        # Convert X to DataFrame to access column names (works even inside ColumnTransformer)
        X_df = pd.DataFrame(X)

        for col in X_df.columns:
            series = X_df[col].dropna()

            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1

            lower = Q1 - self.multiplier * IQR
            upper = Q3 + self.multiplier * IQR

            self.bounds_[col] = (lower, upper)

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)

        for i, col in enumerate(X_df.columns):
            lower, upper = self.bounds_[col]
            X_df.iloc[:, i] = X_df.iloc[:, i].clip(lower=lower, upper=upper)

        return X_df.values

    def get_feature_names_out(self, input_features=None):
        return input_features


In [18]:
data_df = pd.read_csv(TRAIN_CSV_PATH)

# Define lists of column names for different preprocessing steps
ordinal_features = ['income_level']

income_level_order = data_df['income_level'].unique().tolist()

nominal_features = ['country_code', 'country_name', 'region']

uhs_impute_scale_feature = ['uhs_service_coverage_index']

target_columns = [
    'respiratory_disease_rate',
    'cardio_mortality_rate',
    'vector_disease_risk_score',
    'waterborne_disease_incidents',
    'heat_related_admissions'
]

outlier_cols = [
    # Original numerical columns identified for outlier capping and scaling
    'healthcare_access_index', 'mental_health_index', 'food_security_index',
    'pm25_ugm3', 'air_quality_index', 'temperature_celsius', 'temp_anomaly_celsius', 'precipitation_mm',
    # Newly engineered features that may contain outliers and should be capped/scaled
    'pm25_ugm3_lag_1w', 'pm25_ugm3_lag_2w', 'pm25_ugm3_lag_4w', # Lagged pollution
    'temp_lag_1w', 'temp_lag_2w', # Lagged temperature
    'temp_2w_avg', 'temp_2w_volatility', 'temp_4w_volatility', # Rolling temperature stats
    'log_temp_squared', # Transformed temperature
    'temp_change_rate', 'pm25_change_rate', 'precip_change_rate', # Rate of change features
    'spatial_lag_pm25', 'spatial_lag_temp', 'spatial_lag_temp_anomaly', # Spatial lags
    'pm25_temp_interaction', 'temp_precip_interaction', 'pm25_precip_interaction', # Interaction terms
    'extreme_weather_score', # Extreme weather score
    'pollution_vulnerability' # Vulnerability index
]

scale_features = [
    # Original numerical columns to be scaled without specific outlier treatment
    'population_millions', 'aqi_pm', 'gdp_per_capita_usd',
    'latitude', 'longitude', # Geographic coordinates to be scaled
    # Newly engineered features to be scaled
    'day_of_week', 'quarter', # Time-based features
    'distance_to_equator' # Geographic feature
]

passthrough_features = [
    'record_id', # Identifier, should not be transformed
    'country_id',
    # Count/binary features, often kept as-is or handled separately
    'heat_wave_days', 'drought_indicator', 'flood_indicator', 'extreme_weather_events',
    'year', 'date',
    'month_sin', 'month_cos', 'week_sin', 'week_cos', # Cyclic encoded features
    'is_northern', 'is_tropical', # Binary hemisphere indicators
    'log_food_security_index' # Log-transformed, explicitly not to be scaled further
]

# Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal_encoder', OrdinalEncoder(categories=[income_level_order], handle_unknown='use_encoded_value', unknown_value=-1), ordinal_features),
        ('onehot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),

        ('imputer_scaled', Pipeline([
            ('knn', KNNImputer(n_neighbors=5)),
            ('scaler', StandardScaler())
        ]), uhs_impute_scale_feature),

        ('capper_scaled', Pipeline([
            ('capper', IQRCapper(multiplier=1.5)),
            ('scaler', StandardScaler())
        ]), outlier_cols),

        ('num_scaler', StandardScaler(), scale_features),
        ('pass_through', 'passthrough', passthrough_features),
    ],
    remainder='drop'
)

# Assemble all steps into a final pipeline
pipeline = Pipeline([
    ('date_feature_engineer', DateFeatureEngineer()),
    ('preprocessor', preprocessor)
])

X = data_df.drop(columns=target_columns)
y = data_df[target_columns]

# Fit and transform on features values
X_processed_array = pipeline.fit_transform(X)
all_processed_cols = pipeline.get_feature_names_out(input_features=X.columns.tolist())
X_processed = pd.DataFrame(X_processed_array, columns=all_processed_cols, index=data_df.index)

y_scaler = StandardScaler()
y_processed = pd.DataFrame(y_scaler.fit_transform(y), columns=y.columns, index=data_df.index)

print("Pipeline constructed, fitted, and transformed with updated feature lists.")
print(f"Processed DataFrame shape: {X_processed.shape}")
X_processed.fillna(-1, inplace=True)

# Save pipeline
joblib.dump(pipeline, "feature_pipeline.joblib")

# Save Y scaler
joblib.dump(y_scaler, "y_scaler.joblib")

# Save feature names
joblib.dump(all_processed_cols.tolist(), "feature_names.joblib")

f"Feature names saved ({len(all_processed_cols)} features)"

Pipeline constructed, fitted, and transformed with updated feature lists.
Processed DataFrame shape: (14100, 111)


C:\Users\2024\AppData\Local\Temp\ipykernel_20084\3281704745.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_processed.fillna(-1, inplace=True)


'Feature names saved (111 features)'